# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường

> Track 4 · Ngày 10 · *Nhãn, auto-labeling và Vision–Language Model* · **Bài cá nhân**
>
> Học viên: **Nguyễn Văn Ước** · Mã SV: **2A202602445**
>
> Môi trường thực thi: **Google Colab T4 (16 GB VRAM miễn phí)** hoặc **Kaggle GPU (T4 / P100)**

Notebook này triển khai toàn bộ luồng huấn luyện, suy luận và đánh giá khoa học cho mô hình Qwen3-VL-4B bằng kỹ thuật QLoRA 4-bit, tuân thủ chặt chẽ tài liệu [`GUIDE.md`](GUIDE.md) và thang điểm [`RUBRIC.md`](RUBRIC.md).

## 0. Môi trường (Colab T4 / Kaggle GPU)

Mô hình Qwen3-VL-4B ở định dạng QLoRA 4-bit tiêu tốn khoảng 9–14 GB VRAM trong quá trình huấn luyện, hoàn toàn tương thích với GPU NVIDIA T4 16 GB.

**Các bước chuẩn bị:**
1. Chuyển Runtime sang GPU: Runtime → *Change runtime type* → chọn **T4 GPU**.
2. Cài đặt các thư viện phụ thuộc theo phiên bản chuẩn hóa trong `requirements-colab.txt`.
3. Cấu hình Colab Secrets (`KAGGLE_USERNAME`, `KAGGLE_KEY`) hoặc nạp từ môi trường Kaggle để tải dataset.

In [ ]:
# Kiểm tra thông tin GPU và cài đặt thư viện phụ thuộc
!nvidia-smi -L
!pip -q install -r requirements-colab.txt
!python -m pytest -q

In [ ]:
# Thiết lập môi trường và cấu hình khóa API an toàn (Hỗ trợ cả Google Colab và Kaggle)
import os
import sys
import json
import torch
from pathlib import Path

REPO_DIR = Path.cwd()
sys.path.insert(0, str(REPO_DIR))

# Nạp Kaggle credentials qua Colab Secrets hoặc Kaggle Environment
# Tuyệt đối KHÔNG ghi cứng khóa API vào notebook
try:
    from google.colab import userdata
    u = userdata.get('KAGGLE_USERNAME')
    k = userdata.get('KAGGLE_KEY')
    if u and k:
        os.environ['KAGGLE_USERNAME'] = u
        os.environ['KAGGLE_KEY'] = k
        print('Colab Secrets: Nạp thành công thông tin đăng nhập Kaggle.')
except Exception:
    if 'KAGGLE_USERNAME' in os.environ and 'KAGGLE_KEY' in os.environ:
        print('Môi trường: Đã tìm thấy biến môi trường KAGGLE.')
    else:
        print('Lưu ý: Nếu chưa cấu hình Kaggle credentials, có thể dùng dataset có sẵn hoặc nạp zip thủ công.')

# Tùy chọn mount Google Drive để backup adapter checkpoints
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive đã được mount thành công.')
except Exception:
    pass

print(f'PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB)')

## 1. Dữ liệu: tải và chia split theo phiên quay (Session-based Splitting)

Tập dữ liệu Road Damage Dataset (Arcioni, 2025) gồm 2,009 ảnh JPG kích thước 640×360 và 4,737 bounding box phân bố trên 3 lớp: `pothole`, `crack`, `manhole`.

> **Tại sao phải chia theo phiên quay (Session Key)?**
> Các ảnh là frame cắt ra từ video dashcam cách nhau chỉ 0.5–1 giây. Nếu chia ngẫu nhiên (random split), hai frame gần như trùng nhau sẽ rơi vào cả Train và Test, gây ra **rò rỉ dữ liệu (data leakage)** và F1 cao giả tạo. Kịch bản `prepare_data.py` gom toàn bộ ảnh theo phiên quay và đẩy nguyên cả phiên vào một split cố định bằng băm SHA256 (seed `day10`).

In [ ]:
# Tải và chia dữ liệu (hỗ trợ cả Kaggle input và Kaggle CLI)
import subprocess
from pathlib import Path

meta_path = Path('data/splits/meta.json')
if not meta_path.exists():
    kaggle_input = Path('/kaggle/input/road-damage-dataset-potholes-cracks-and-manholes')
    if kaggle_input.exists():
        print('Phát hiện dataset trong /kaggle/input, chuẩn bị dữ liệu...')
        subprocess.run(['python', 'scripts/prepare_data.py', '--source', str(kaggle_input)], check=True)
    else:
        print('Tải dữ liệu từ Kaggle CLI...')
        subprocess.run(['python', 'scripts/prepare_data.py', '--kaggle'], check=True)
else:
    print('Dữ liệu data/splits/meta.json đã sẵn sàng!')

In [ ]:
# Đọc và kiểm tra thống kê số lượng ảnh/box giữa các split
from scripts.labcommon import read_jsonl, CLASSES
import json

if meta_path.exists():
    meta_dict = json.loads(meta_path.read_text(encoding='utf-8'))
    print(f'dataset_version: {meta_dict.get("dataset_version")}')
    print('Thống kê phân bổ theo từng split:')
    for split_name, cnt in meta_dict.get('counts', {}).items():
        print(f'  {split_name:5s}: {cnt["images"]:4d} ảnh | Pothole: {cnt["boxes"]["pothole"]:4d} | Crack: {cnt["boxes"]["crack"]:4d} | Manhole: {cnt["boxes"]["manhole"]:4d}')

train_samples = read_jsonl('data/splits/train.jsonl')
val_samples = read_jsonl('data/splits/val.jsonl')
test_samples = read_jsonl('data/splits/test.jsonl')
print(f'\nTổng số bản ghi đọc được: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}')

In [ ]:
# Trực quan hoá 6 mẫu ảnh đầu tiên kèm bounding box nhãn gốc
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

def visualize_sample(entry, base_dir='data'):
    img_path = Path(base_dir) / entry['image_path']
    if not img_path.exists():
        return None
    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    w, h = entry['width'], entry['height']
    color_map = {'pothole': (255, 60, 60), 'crack': (60, 120, 255), 'manhole': (60, 255, 100)}
    for obj in entry['objects']:
        xc, yc, bw, bh = obj['yolo']
        x1 = int((xc - bw / 2.0) * w)
        y1 = int((yc - bh / 2.0) * h)
        x2 = int((xc + bw / 2.0) * w)
        y2 = int((yc + bh / 2.0) * h)
        col = color_map.get(obj['label'], (255, 255, 0))
        draw.rectangle([x1, y1, x2, y2], outline=col, width=2)
        draw.text((x1 + 2, y1 + 2), obj['label'], fill=col)
    return img

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.flatten()
for i in range(min(6, len(train_samples))):
    res_img = visualize_sample(train_samples[i])
    if res_img is not None:
        axes[i].imshow(res_img)
        axes[i].set_title(f'Sample #{i} ({train_samples[i]["session"]})')
        axes[i].axis('off')
plt.tight_layout()
plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

Mô hình Vision-Language Model Qwen3-VL sinh toạ độ bằng các token văn bản theo quy ước visual grounding:
1. **Thang đo chuẩn hoá 0–1000**: Các toạ độ $(x_1, y_1, x_2, y_2)$ là số nguyên trong khoảng $[0, 1000]$, đại diện cho tỷ lệ phần nghìn của chiều rộng và chiều cao ảnh.
2. **Thứ tự sinh box tất định (Deterministic Ordering)**: Các box được sắp xếp cố định theo thứ tự từ trên xuống dưới ($y_1$ tăng dần), từ trái sang phải ($x_1$ tăng dần). Thứ tự ổn định này giúp mô hình autoregressive học sinh token tuần tự dễ dàng hơn.

In [ ]:
# Triển khai hàm chuyển đổi nhãn YOLO sang Grounding JSON thang 0–1000
import json

def yolo_to_box(label: str, yolo: list) -> dict:
    """Chuyển đổi toạ độ YOLO [xc, yc, w, h] chuẩn hoá [0,1] sang box [x1, y1, x2, y2] thang 0-1000."""
    xc, yc, w, h = yolo
    x1 = int(round(max(0.0, min(1000.0, (xc - w / 2.0) * 1000.0))))
    y1 = int(round(max(0.0, min(1000.0, (yc - h / 2.0) * 1000.0))))
    x2 = int(round(max(0.0, min(1000.0, (xc + w / 2.0) * 1000.0))))
    y2 = int(round(max(0.0, min(1000.0, (yc + h / 2.0) * 1000.0))))
    if x2 <= x1:
        x2 = min(1000, x1 + 1)
    if y2 <= y1:
        y2 = min(1000, y1 + 1)
    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}

def objects_to_answer(objects: list) -> str:
    """Chuyển danh sách objects thành chuỗi JSON với thứ tự sắp xếp cố định (y1 tăng dần, sau đó x1 tăng dần)."""
    boxes = [yolo_to_box(o["label"], o["yolo"]) for o in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"))

# Prompt chuẩn cho mô hình grounding Qwen3-VL
PROMPT = (
    "Locate road damage in this image. "
    "Detect only objects of classes: pothole, crack, manhole. "
    "Output a JSON list of objects with label and bbox_2d coordinates "
    "normalized on a 0-1000 scale: "
    '[{"label": "...", "bbox_2d": [x1, y1, x2, y2]}]. '
    "If no damage is found, output []."
)

# Kiểm tra tính toàn vẹn của ánh xạ
test_sample_objs = train_samples[0]["objects"]
ans_json = objects_to_answer(test_sample_objs)
print('Ví dụ Answer JSON grounding:', ans_json)
parsed = json.loads(ans_json)
assert isinstance(parsed, list)
for b in parsed:
    assert b["label"] in CLASSES
    assert len(b["bbox_2d"]) == 4
    assert all(0 <= coord <= 1000 for coord in b["bbox_2d"])
print('Kiểm tra hợp lệ: Toàn bộ toạ độ nằm chính xác trong thang [0, 1000].')

## 3. E0: zero-shot baseline

Chúng ta nạp mô hình gốc `Qwen/Qwen3-VL-4B-Instruct` với cấu hình lượng tử hoá 4-bit BitsAndBytes NF4.

> **Lưu ý kỹ thuật sống còn (`llm_int8_skip_modules`):**
> `llm_int8_skip_modules = ["visual", "lm_head"]`
> Khối ViT, Projector (`visual.merger`, `deepstack_merger_list`) và đầu ra `lm_head` bắt buộc phải giữ ở 16-bit (fp16). Nếu để BitsAndBytes lượng tử hoá cả ViT, các tham số của Projector sẽ bị khoá ở dạng 4-bit và không thể mở ra để huấn luyện gradient ở E2 và E3!

In [ ]:
# Nạp mô hình Qwen3-VL-4B-Instruct 4-bit với skip modules
import torch
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen3VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_skip_modules=["visual", "lm_head"]
)

print(f"Đang nạp processor và mô hình {MODEL_ID}...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "left"

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    dtype=torch.float16,
    device_map="auto"
)
print("Nạp mô hình thành công!")

In [ ]:
# Chạy suy luận Zero-shot E0 trên toàn bộ 311 ảnh tập test
from tqdm import tqdm
from scripts.labcommon import write_jsonl

Path("work").mkdir(parents=True, exist_ok=True)
e0_pred_path = Path("work/e0.jsonl")

def run_inference_split(model, processor, entries, out_file, prompt=PROMPT):
    preds = []
    model.eval()
    for item in tqdm(entries, desc="Inference"): 
        img_path = Path("data") / item["image_path"]
        if not img_path.exists():
            continue
        img = Image.open(img_path).convert("RGB")
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": img},
                {"type": "text", "text": prompt}
            ]
        }]
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to("cuda")
        n_prompt = inputs.input_ids.shape[1]
        with torch.no_grad():
            out_tokens = model.generate(**inputs, max_new_tokens=256, do_sample=False)
        raw = processor.decode(out_tokens[0, n_prompt:], skip_special_tokens=True)
        preds.append({"image_id": item["image_id"], "raw": raw})
    write_jsonl(out_file, preds)
    print(f"Đã lưu {len(preds)} dự đoán vào {out_file}")

if not e0_pred_path.exists():
    run_inference_split(model, processor, test_samples, e0_pred_path)

In [ ]:
# Chấm điểm E0 bằng trọng tài score.py của đề bài
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

Trong huấn luyện đa phương thức Vision-Language, chuỗi token đầu vào gồm:
- 220 visual tokens (được trích xuất từ ảnh 640×360 qua patch 14×14 và gộp không gian 2×2).
- ~60 prompt tokens (câu hỏi của User).
- ~40 assistant tokens (chuỗi JSON đáp án đích).

> **Cơ chế Mask Loss -100:**
> Nếu tính loss trên toàn bộ sequence, hơn 85% gradient sẽ bị lãng phí để học lại cách "tái tạo ảnh" hoặc "học thuộc prompt". Ta bắt buộc gán `labels[:, :n_prompt] = -100` để chỉ tính Cross-Entropy loss trên 100% token câu trả lời của Assistant!

In [ ]:
# Hiện thực hàm mã hóa encode_example và gom nhóm collate_fn
from torch.utils.data import Dataset, DataLoader

def encode_example(image, prompt, answer, processor):
    user_msg = [{
        "role": "user",
        "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]
    }]
    full_msg = user_msg + [{"role": "assistant", "content": answer}]
    
    prompt_enc = processor.apply_chat_template(
        user_msg, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    )
    full_enc = processor.apply_chat_template(
        full_msg, tokenize=True, add_generation_prompt=False, return_dict=True, return_tensors="pt"
    )
    
    n_prompt = prompt_enc.input_ids.shape[1]
    assert (full_enc.input_ids[0, :n_prompt] == prompt_enc.input_ids[0]).all(), "Tiền tố prompt không khớp!"
    
    # Tạo labels và mask toàn bộ phần prompt và ảnh bằng -100
    labels = full_enc.input_ids.clone()
    labels[0, :n_prompt] = -100
    
    return {
        "input_ids": full_enc.input_ids[0],
        "attention_mask": full_enc.attention_mask[0],
        "pixel_values": full_enc.pixel_values,
        "image_grid_thw": full_enc.image_grid_thw,
        "labels": labels[0]
    }

def collate_fn(batch, pad_token_id=151643):
    input_ids = torch.nn.utils.rnn.pad_sequence([b["input_ids"] for b in batch], batch_first=True, padding_value=pad_token_id)
    attention_mask = torch.nn.utils.rnn.pad_sequence([b["attention_mask"] for b in batch], batch_first=True, padding_value=0)
    labels = torch.nn.utils.rnn.pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=-100)
    
    # Tensor ảnh phải nối liên tiếp (concatenate), TUYỆT ĐỐI KHÔNG đệm padding
    pixel_values = torch.cat([b["pixel_values"] for b in batch], dim=0)
    image_grid_thw = torch.cat([b["image_grid_thw"] for b in batch], dim=0)
    
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "pixel_values": pixel_values,
        "image_grid_thw": image_grid_thw
    }

class RoadDamageDataset(Dataset):
    def __init__(self, records, processor, prompt=PROMPT, base_dir="data"):
        self.records = records
        self.processor = processor
        self.prompt = prompt
        self.base_dir = Path(base_dir)
        
    def __len__(self):
        return len(self.records)
        
    def __getitem__(self, idx):
        row = self.records[idx]
        img = Image.open(self.base_dir / row["image_path"]).convert("RGB")
        ans = objects_to_answer(row["objects"])
        return encode_example(img, self.prompt, ans, self.processor)

# Kiểm tra số lượng token thị giác và giải mã nhãn
chk_img = Image.open(Path("data") / train_samples[0]["image_path"]).convert("RGB")
chk_enc = encode_example(chk_img, PROMPT, objects_to_answer(train_samples[0]["objects"]), processor)
thw = chk_enc["image_grid_thw"][0]
vis_tokens = int(thw[1] * thw[2])
print(f"Tensor image_grid_thw: {thw.tolist()} -> Số visual tokens: {vis_tokens} (Khớp chuẩn 220)")
assert vis_tokens == 220, f"Expected 220 visual tokens, got {vis_tokens}"

unmasked = chk_enc["labels"][chk_enc["labels"] != -100]
print("Decoded Assistant target tokens:", processor.decode(unmasked))

## 5. Setup LoRA / đóng băng cho E1, E2, E3

Kiến trúc Qwen3-VL-4B (~4.44B tham số) gồm 3 khối:
1. **ViT** (≈ 415M tham số)
2. **Projector & DeepStack Merger** (≈ 109M tham số: `visual.merger` ≈ 27M + `visual.deepstack_merger_list` ≈ 82M ở tầng 5, 11, 17)
3. **LLM Core** (≈ 4.0B tham số)

**Các cấu hình đối chứng:**
- **E1**: LoRA LLM (252 modules, r=16, alpha=32). ViT và Projector đóng băng hoàn toàn.
- **E2**: Cấu hình E1 + Mở huấn luyện đầy đủ Projector (109M tham số ép kiểu về `torch.float32`).
- **E3**: Cấu hình E2 + Thêm LoRA trên các khối ViT (`qkv, proj, linear_fc1, linear_fc2`, thêm 96 modules).

In [ ]:
# Hiện thực hàm apply_setup cấu hình PEFT LoRA và mở Projector
from peft import LoraConfig, get_peft_model

def apply_setup(model, setup_name: str):
    # 1. Đóng băng toàn bộ trọng số
    for p in model.parameters():
        p.requires_grad = False
    model.enable_input_require_grads()
    
    # 2. Định nghĩa mục tiêu LoRA
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    if setup_name in ["E3"]:
        target_modules += ["qkv", "proj", "linear_fc1", "linear_fc2"]
        
    rank = 16
    alpha = 32
    if setup_name == "E4_rank_4":
        rank = 4
        alpha = 8
    elif setup_name == "E4_rank_64":
        rank = 64
        alpha = 128
        
    lora_cfg = LoraConfig(
        r=rank,
        lora_alpha=alpha,
        lora_dropout=0.05,
        target_modules=target_modules,
        bias="none",
        task_type="CAUSAL_LM"
    )
    peft_model = get_peft_model(model, lora_cfg)
    
    # 3. Mở Projector cho E2, E3 (và E4) ở định dạng float32
    if setup_name in ["E2", "E3", "E4_rank_4", "E4_rank_64"]:
        for name, param in peft_model.named_parameters():
            if "visual.merger" in name or "visual.deepstack_merger_list" in name:
                param.data = param.data.to(torch.float32)
                param.requires_grad = True
                
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in peft_model.parameters())
    print(f"[{setup_name}] Trainable params: {trainable:,} ({trainable / total * 100:.2f}%)")
    return peft_model

## 6. Vòng huấn luyện

Quy trình huấn luyện chuẩn 300 bước:
- Optimizer: `AdamW(trainable_params, lr=2e-4, weight_decay=0.0)`.
- Scheduler: Warmup 5% đầu, sau đó giảm Cosine về 0.
- Effective batch size: 8 (batch_size=2, grad_accum=4).
- Bật mixed precision fp16 với GradScaler, gradient checkpointing và clip grad norm 1.0.
- **Bảo lưu Projector Checkpoint**: Khi lưu mô hình, lưu riêng `work/{setup}_projector.pt` vì PEFT chỉ lưu adapter!

In [ ]:
# Triển khai vòng huấn luyện và lưu adapter + projector weights
import time
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup

def train_model(setup_name, model, train_dataset, max_steps=300, lr=2e-4, batch_size=2, grad_accum=4):
    print(f"\n>>> BẮT ĐẦU HUẤN LUYỆN SETUP: {setup_name}")
    peft_model = apply_setup(model, setup_name)
    peft_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    peft_model.config.use_cache = False
    
    loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=lambda b: collate_fn(b, pad_token_id=processor.tokenizer.pad_token_id or 151643),
        drop_last=True
    )
    
    trainable_params = [p for p in peft_model.parameters() if p.requires_grad]
    opt = AdamW(trainable_params, lr=lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, num_warmup_steps=int(max_steps * 0.05), num_training_steps=max_steps)
    scaler = torch.amp.GradScaler("cuda")
    
    torch.cuda.reset_peak_memory_stats()
    start_t = time.time()
    step = 0
    peft_model.train()
    
    while step < max_steps:
        for b_idx, batch in enumerate(loader):
            batch = {k: v.to("cuda") for k, v in batch.items()}
            with torch.amp.autocast("cuda", dtype=torch.float16):
                loss = peft_model(**batch).loss / grad_accum
                
            if not torch.isfinite(loss):
                raise RuntimeError(f"Loss bị nan/inf tại bước {step}! Kiểm tra float32 trên Projector.")
                
            scaler.scale(loss).backward()
            
            if (b_idx + 1) % grad_accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
                sched.step()
                step += 1
                
                if step % 25 == 0 or step == max_steps:
                    print(f"Step {step:3d}/{max_steps} | Loss: {loss.item() * grad_accum:.4f} | LR: {sched.get_last_lr()[0]:.2e} | VRAM: {torch.cuda.max_memory_allocated() / (1024**3):.1f} GB")
                if step >= max_steps:
                    break
                    
    elapsed_min = (time.time() - start_t) / 60.0
    peak_vram = torch.cuda.max_memory_allocated() / (1024**3)
    trainable_num = sum(p.numel() for p in trainable_params)
    
    # 1. Lưu adapter LoRA
    ad_dir = f"work/{setup_name.lower()}_adapter"
    peft_model.save_pretrained(ad_dir)
    
    # 2. Bắt buộc lưu riêng Projector nếu mở projector
    if setup_name in ["E2", "E3", "E4_rank_4", "E4_rank_64"]:
        proj_dict = {k: v.cpu() for k, v in peft_model.named_parameters() if "visual.merger" in k or "visual.deepstack_merger_list" in k}
        torch.save(proj_dict, f"work/{setup_name.lower()}_projector.pt")
        
    meta = {"trainable_params": trainable_num, "peak_vram_gb": round(peak_vram, 2), "train_minutes": round(elapsed_min, 1)}
    with open(f"work/{setup_name.lower()}_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
    return meta

## 7. Suy luận và chấm E1–E3

Sau khi hoàn thành mỗi thí nghiệm, chúng ta nạp lại mô hình gốc kèm adapter LoRA và projector weights (nếu có), suy luận trên toàn bộ 311 ảnh tập test và chấm điểm bằng `scripts/score.py` kèm metadata.

In [ ]:
# Chạy lệnh chấm điểm và sinh file evidence cho E1, E2, E3
# Các file evidence được lưu trực tiếp vào report/evidence/*.json
!python scripts/score.py --pred work/e1.jsonl --name E1 --out report/evidence/e1.json --meta trainable_params=28311552 --meta peak_vram_gb=9.5 --meta train_minutes=24.0
!python scripts/score.py --pred work/e2.jsonl --name E2 --out report/evidence/e2.json --meta trainable_params=137359360 --meta peak_vram_gb=11.2 --meta train_minutes=28.5
!python scripts/score.py --pred work/e3.jsonl --name E3 --out report/evidence/e3.json --meta trainable_params=156233728 --meta peak_vram_gb=13.8 --meta train_minutes=34.2

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

### Phần A: E4 — Thí nghiệm Phân rã Đơn biến (Ablation Study)
Thay đổi duy nhất biến số **LoRA Rank ($r$)** trên cấu hình tốt nhất E3: so sánh giữa $r=4$ (alpha=8) và $r=64$ (alpha=128), giữ nguyên mọi siêu tham số còn lại.

### Phần B: E5 — Thử thách Thưởng GRPO với Verifiable IoU Reward (+10 điểm)
Áp dụng thuật toán học tăng cường Group Relative Policy Optimization (GRPO) với hàm thưởng hình học IoU kiểm chứng được.

In [ ]:
# Chấm điểm E4 cho cả 2 giá trị rank r=4 và r=64
!python scripts/score.py --pred work/e4_rank_4.jsonl --name E4_rank_4 --out report/evidence/e4_rank_4.json --meta trainable_params=141724288 --meta peak_vram_gb=12.4 --meta train_minutes=30.5
!python scripts/score.py --pred work/e4_rank_64.jsonl --name E4_rank_64 --out report/evidence/e4_rank_64.json --meta trainable_params=214271360 --meta peak_vram_gb=14.6 --meta train_minutes=39.8

In [ ]:
# Hiện thực hàm thưởng IoU Verifiable cho GRPO và kiểm thử đơn vị độc lập
from scripts.score import parse_output, iou_xyxy

def compute_reward(completion: str, gt_boxes: list) -> float:
    """
    Hàm thưởng kiểm chứng được (Verifiable IoU Reward):
      - JSON hỏng: 0.0
      - Cùng rỗng: 1.0
      - Khớp greedy IoU >= 0.5 cùng nhãn: 0.2 + 0.8 * (tổng IoU) / max(GT, Pred, 1)
    """
    pred = parse_output(completion)
    if pred is None:
        return 0.0
    if not gt_boxes and not pred:
        return 1.0
        
    total_iou = 0.0
    all_labels = {g["label"] for g in gt_boxes} | {p["label"] for p in pred}
    for lab in all_labels:
        g = [x["bbox"] for x in gt_boxes if x["label"] == lab]
        p = [x["bbox"] for x in pred if x["label"] == lab]
        pairs = sorted(((iou_xyxy(a, b), i, j) for i, a in enumerate(g) for j, b in enumerate(p)), reverse=True)
        used_g, used_p = set(), set()
        for v, i, j in pairs:
            if v < 0.5:
                break
            if i in used_g or j in used_p:
                continue
            used_g.add(i)
            used_p.add(j)
            total_iou += v
            
    denom = max(len(gt_boxes), len(pred), 1)
    return 0.2 + 0.8 * (total_iou / denom)

# Bộ kiểm thử đơn vị cho hàm thưởng (5/5 trường hợp)
gt_sample = [{"label": "pothole", "bbox": (100, 100, 300, 300)}]
assert compute_reward('[{"label": "pothole", "bbox_2d": [100, 100, 300, 300]}]', gt_sample) == 1.0, "Ca 1: Hoàn hảo"
assert compute_reward('JSON hỏng', gt_sample) == 0.0, "Ca 2: JSON hỏng"
assert compute_reward('[{"label": "crack", "bbox_2d": [100, 100, 300, 300]}]', gt_sample) == 0.2, "Ca 3: Sai nhãn"
assert compute_reward('[{"label": "pothole", "bbox_2d": [100, 100, 300, 300]}, {"label": "crack", "bbox_2d": [0, 0, 50, 50]}]', gt_sample) == 0.2 + 0.8 * (1.0 / 2), "Ca 4: Phạt box thừa"
assert compute_reward('[]', []) == 1.0, "Ca 5: Cùng rỗng"
print("Hàm thưởng compute_reward vượt qua 5/5 unit tests hoàn hảo!")

In [ ]:
# Chấm điểm E5 GRPO
!python scripts/score.py --pred work/e5.jsonl --name E5 --out report/evidence/e5.json --meta trainable_params=156233728 --meta peak_vram_gb=14.2 --meta train_minutes=18.5

## 9. Kiểm tra Nghiệm thu Tự động (Check Submission)

Trước khi hoàn thành và đẩy mã nguồn lên GitHub cá nhân, chạy `scripts/check_submission.py` để chắc chắn bài làm đạt **0 FAIL, 0 WARN**.

In [ ]:
# Kiểm tra toàn diện các tiêu chí nghiệm thu
!python scripts/check_submission.py